In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 샘플링과 이산 푸리에 변환 (DFT)

## 학습 목표
- 샘플링 주파수 $f_s$ 와 샘플링 정리, 에일리어싱(aliasing)을 실험으로 확인한다.
- DFT 정의식을 직접 구현하고 `np.fft.fft` 와 비교한다.
- 양측 스펙트럼과 단측 스펙트럼, 진폭 스케일링을 다룬다.
- 합성 신호 CSV 파일에 FFT 를 적용한다.

## 환경 설정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from utils import fft

---

# Part A. 샘플링

컴퓨터가 연속 신호 $x(t)$ 를 다루려면 일정한 간격 $T_s = 1/f_s$ 마다 값을 읽어 **이산 수열** $x[n] = x(nT_s)$ 로 변환해야 한다. 이 과정을 **샘플링**이라고 한다.

- 샘플링 주파수: $f_s$(Hz, 초당 샘플 수)
- 샘플 간격: $T_s = 1/f_s$(초)
- $N$개 샘플의 신호 길이: `duration = N/fs`

## 실습 1. 연속 신호의 샘플링 기본

$f_0 = 3$ Hz 의 코사인 신호를 $f_s = 30$ Hz 로 샘플링한다. 연속 신호는 부드러운 실선으로, 이산 샘플은 `stem` 으로 표시한다.

In [ ]:
f0 = 3            # 신호 주파수 (Hz)
fs = 30           # 샘플링 주파수 (Hz)
duration = 1.0    # 신호 길이 (초)

# 연속 신호 근사 (매우 조밀한 샘플링)
t_cont = np.linspace(0, duration, 5000)
x_cont = np.cos(2*np.pi*f0*t_cont)

# 샘플링
N = int(fs * duration)
n = np.arange(N)
t = n / fs
x = np.cos(2*np.pi*f0*t)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_cont, x_cont, color='C0', alpha=0.5, label=f'Continuous Signal {f0} Hz')
ax.stem(t, x, linefmt='C1-', markerfmt='C1o', basefmt=' ', label=f'Samples ($f_s$={fs} Hz, N={N})')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Sampling of Continuous Signal')
ax.legend()
plt.tight_layout()
plt.show()

- $f_s = 30$ Hz 는 Nyquist 조건 $f_s > 2 f_0 = 6$ Hz 를 충분히 만족한다.
- 샘플 간격은 $T_s = 1/30 \approx 0.033$ s 이고, 1초 동안 30개의 샘플을 얻는다.
- 샘플만 보아도 원래 3 Hz 코사인 신호의 형태를 그대로 알아볼 수 있다.

### 직접 해보기
1. `fs = 10` 으로 낮추면 샘플은 몇 개가 되는가? 한 주기에 샘플이 몇 개 놓이며, 원 신호의 형태를 여전히 알아볼 수 있는가?
2. `fs = 10` 은 그대로 두고 `f0` 를 1, 2, 5, 9 Hz 로 바꾸어 가며 실행해 보라. $f_0 = 5$ Hz($= f_s/2$)에서 샘플은 어떻게 놓이는가? $f_0 = 9$ Hz 의 샘플은 몇 Hz 코사인의 샘플과 같은 값을 갖는가?
3. `fs = 30` 으로 되돌리고 `f0 = 20` 으로 바꾸면 샘플은 몇 Hz 코사인처럼 보이는가? 이 현상은 실습 2에서 다룬다.

---

## 실습 2. Nyquist 주파수와 에일리어싱

**샘플링 정리**: $f_s > 2 f_{\max}$ 이어야 원 신호를 복원할 수 있다. 이 조건을 만족하지 않으면 고주파 성분이 낮은 주파수의 **에일리어스**(alias)로 나타난다.

$$ f_{\text{alias}} = | f_0 - k\,f_s |, \quad k = \text{round}(f_0 / f_s) $$

$f_0 = 80$ Hz 의 신호를 두 가지 샘플링 주파수로 샘플링하여 비교한다.

| $f_s$ (Hz) | $f_s/2$ | 조건 | 결과 |
|:---:|:---:|:---:|:---|
| 200 | 100 | $100 > 80$ 충족 | 정상 샘플링 |
| 100 | 50  | $50 < 80$ 위반 | 에일리어스 $|80-100| = 20$ Hz |

In [ ]:
f0 = 80
duration = 0.1
t_cont = np.linspace(0, duration, 8000)
x_cont = np.cos(2*np.pi*f0*t_cont)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for ax, fs in zip(axes, [200, 100]):
    N = int(fs * duration)
    t = np.arange(N) / fs
    x = np.cos(2*np.pi*f0*t)

    ax.plot(t_cont*1000, x_cont, color='C0', alpha=0.4, label=f'Original {f0} Hz')
    ax.stem(t*1000, x, linefmt='C1-', markerfmt='C1o', basefmt=' ', label='Samples')
    ax.set_title(f'$f_s$ = {fs} Hz')
    ax.set_ylabel('Amplitude')
    ax.set_ylim([-1.4, 1.4])
    ax.legend(loc='upper right')

# 아래 패널 (fs = 100 Hz): 같은 샘플을 지나는 에일리어스 코사인
f_alias = abs(f0 - 100)
axes[1].plot(t_cont*1000, np.cos(2*np.pi*f_alias*t_cont), 'C2--', label=f'Alias {f_alias} Hz')
axes[1].legend(loc='upper right')

axes[-1].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

- 위 패널($f_s = 200$)에서는 $f_s/2 = 100$ Hz 가 80 Hz 보다 크므로 조건을 만족한다. 한 주기에 샘플이 2.5개뿐이므로 샘플만 이어서는 80 Hz 파형의 모양을 알아볼 수 없다. 그러나 샘플 20개는 모두 80 Hz 곡선 위에 있으며, 0 ~ 100 Hz 범위에서 이 샘플을 지나는 코사인은 80 Hz 하나뿐이다.
- 아래 패널($f_s = 100$)에서는 샘플이 파선으로 그린 20 Hz 코사인 위에도 그대로 놓이므로 두 신호를 구분할 수 없다. 이것이 **에일리어싱**(aliasing)이다.
- 에일리어싱이 발생하면 스펙트럼에도 실제로는 존재하지 않는 20 Hz 성분이 나타난다. 스펙트럼에서 에일리어스가 나타나는 모습은 다음 실습 3에서 다른 신호로 확인한다.

### 직접 해보기
1. Nyquist 조건의 경계인 `fs = 160`($f_s/2 = 80 = f_0$)으로 실행하면 샘플 값은 어떻게 되는가?
2. `f0 = 60, fs = 100` 에서 에일리어스 주파수를 계산하고 그래프로 확인한다.

---

## 실습 3. $f_s$ 의 단계적 변화와 샘플링 정리

$f_{\max} = 120$ Hz 인 합성 신호를 세 가지 $f_s$ 로 샘플링하고, 시간 영역뿐 아니라 주파수 영역에서도 비교한다.

$$x(t) = \cos(2\pi\cdot 50 t) + 0.7\cos(2\pi\cdot 120 t)$$

| $f_s$ | $f_s/2$ | 조건 | 예상 결과 |
|:---:|:---:|:---:|:---|
| 500 | 250 | 충족(여유 130 Hz) | 50, 120 Hz 성분이 선명하게 나타남 |
| 300 | 150 | 충족(여유 30 Hz) | 50, 120 Hz 성분이 나타남 |
| 200 | 100 | 위반 | 120 Hz 성분이 80 Hz 에 나타남 |

> 아래의 스펙트럼은 헬퍼 함수 `utils.fft` 로 계산한다. 이 함수의 내부 정의와 사용 규약은 실습 9에서 자세히 다룬다.

In [ ]:
f_max = 120
duration = 0.5  # 전체 신호 길이 (초)

# 원 신호 (연속 파형): 왼쪽 패널의 40 ms 구간에 겹쳐 그림
t_cont = np.linspace(0, 0.04, 4000)
x_cont = np.cos(2*np.pi*50*t_cont) + 0.7*np.cos(2*np.pi*120*t_cont)

fig, axes = plt.subplots(3, 2, figsize=(14, 8))

for row, fs in enumerate([500, 300, 200]):
    N = int(fs * duration)
    t = np.arange(N) / fs
    x = np.cos(2*np.pi*50*t) + 0.7*np.cos(2*np.pi*120*t)

    axes[row, 0].plot(t_cont*1000, x_cont, color='C0', alpha=0.4, label='Original')
    axes[row, 0].plot(t*1000, x, '-o', color='C1', markersize=4, linewidth=1.0, label='Samples')
    axes[row, 0].set_xlim([0, 40])
    axes[row, 0].set_ylim([-2.0, 2.0])
    axes[row, 0].set_title(f'$f_s$={fs} Hz — Time Domain')
    axes[row, 0].set_xlabel('Time (ms)')
    axes[row, 0].set_ylabel('Amplitude')
    axes[row, 0].legend(loc='upper right', fontsize=8)

    freq, amp = fft(x, fs)
    axes[row, 1].plot(freq, amp, color='C1', lw=1.2)
    axes[row, 1].set_title(f'$f_s$={fs} Hz — Spectrum')
    axes[row, 1].set_xlabel('Frequency (Hz)')
    axes[row, 1].set_ylabel('Amplitude')
    axes[row, 1].set_xlim([0, 260])
    axes[row, 1].set_ylim([0, 1.3])
    axes[row, 1].axvline(fs/2, color='C0', ls=':', alpha=0.7, label=f'$f_s/2$ = {fs/2:.0f} Hz')
    axes[row, 1].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

- $f_s = 500, 300$ 은 모두 Nyquist 조건을 만족하므로 50 Hz 와 120 Hz 에 각각 원 신호의 진폭과 같은 크기의 성분이 나타난다.
- $f_s = 200$ 에서는 120 Hz 성분이 $|120 - 200| = 80$ Hz 에서 관측된다. 스펙트럼만으로는 80 Hz 에 실제로 성분이 있는지, 120 Hz 성분의 에일리어스인지 구별할 수 없다.

### 직접 해보기
1. 신호에 `+ 0.5*np.cos(2*np.pi*80*t)` 성분을 추가한다. $f_s = 200$ 에서 80 Hz 성분의 크기는 두 성분이 더해져 커지는가, 작아지는가? 결과는 위상에 따라 다르다.
2. $f_s = 250$ 에서 실행하면 120 Hz 성분은 에일리어싱 없이 나타나지만 여유가 5 Hz 뿐이다. 스펙트럼에 두 성분 외의 값이 나타나는지 확인한다.

---

# Part B. 이산 푸리에 변환 (DFT)

샘플링으로 얻은 이산 수열 $x[n]$($n = 0, 1, \ldots, N-1$)의 주파수 스펙트럼을 유한 합으로 계산하는 것이 **이산 푸리에 변환**(DFT)이다.

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j 2\pi k n / N}, \qquad k = 0, 1, \ldots, N-1$$

빈 인덱스 $k$ 는 주파수 $f_k = k\,f_s/N$ 에 대응한다. 주파수 분해능은 $\Delta f = f_s/N = 1/T$ 이다.

## 실습 4. 공통 테스트 신호

실습 5 ~ 7과 실습 9에서 사용할 간단한 합성 신호는 다음과 같다.
$x(t) = \cos(2\pi f_0 t) + 0.5\,\cos(2\pi f_1 t)$

설정은 `fs=40`, `N=10`, `f_0=4` Hz, `f_1=12` Hz 이다. $\Delta f = f_s/N = 4$ Hz 이므로 두 주파수가 모두 빈의 중심에 정확히 놓여 누설이 발생하지 않는다.

In [ ]:
fs, N = 40, 10
f0, f1 = 4, 12
n = np.arange(N)
t = n / fs
x = np.cos(2*np.pi*f0*t) + 0.5*np.cos(2*np.pi*f1*t)
print(f'fs={fs} Hz, N={N}, T={N/fs:.3f} s, df={fs/N:.2f} Hz')

**관찰**: 기본 설정은 $x(t) = \cos(2\pi \cdot 4 t) + 0.5\cos(2\pi \cdot 12 t)$, $N=10$, $f_s=40$ Hz 이다. 이후의 실습에서는 이 신호의 DFT, FFT, 단측 스펙트럼, 양측 스펙트럼을 반복하여 사용한다.

---

## 실습 5. DFT 공식 직접 구현 (O(N²))

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j\,2\pi k n / N}, \qquad k = 0, 1, \ldots, N-1$$

행렬 $W_{k,n} = e^{-j\,2\pi k n / N}$ 을 만들어 $X = W\, x$ 로 한 번에 계산한다.

In [ ]:
def dft_manual(x, fs):
    """DFT 공식을 행렬 곱으로 직접 구현 (O(N^2)).

    주파수 인덱스는 fftfreq 관례로 반환: k < ceil(N/2) 는 양의 주파수,
    k >= ceil(N/2) 는 Nyquist 주파수 이상이므로 음의 주파수로 읽는다.

    Returns
    -------
    freq : ndarray (N,)  -fs/2 ~ +fs/2 (fftfreq 관례, raw 순서)
    X    : ndarray (N,)  복소 스펙트럼
    """
    x = np.asarray(x, dtype=complex)
    N = len(x)
    n = np.arange(N)                    # 시간 인덱스 (열)
    k = np.arange(N).reshape(N, 1)      # 주파수 인덱스 (행)
    W = np.exp(-1j * 2 * np.pi * k * n / N)   # W[k, n]
    X = W @ x
    # k >= ceil(N/2) 는 Nyquist 주파수 이상이므로 음의 주파수로 표기
    kk = np.arange(N)
    freq = np.where(kk < (N + 1) // 2, kk, kk - N) * fs / N
    return freq, X

freq_manual, X_manual = dft_manual(x, fs)

print('k         :', np.arange(len(x)))
print('freq (Hz) :', freq_manual)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.stem(freq_manual, np.abs(X_manual), linefmt='C0-', markerfmt='C0o', basefmt=' ')
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('|X|')
ax.set_title('dft_manual — magnitude spectrum (fftfreq convention)')
fig.tight_layout(); plt.show()

**관찰**: 행렬 $W_{k,n} = e^{-j 2\pi kn/N}$ 을 구성하고 `W @ x` 로 DFT 를 직접 계산했다. 반환하는 주파수 축은 `np.fft.fftfreq` 의 관례를 따르므로, 인덱스가 $k \geq \lceil N/2 \rceil$ 인 빈은 음의 주파수에 대응한다.

---

## 실습 6. 빠른 대안인 NumPy FFT 함수군

DFT 공식을 그대로 구현하면 결과는 옳지만 연산량이 O(N²) 이므로 느리다. NumPy 는 같은 결과를 **O(N log N)** 으로 얻는 **FFT** 와 함께, 주파수 축을 다루는 함수도 제공한다.

| 함수 | 역할 |
|---|---|
| `np.fft.fft(x)` / `np.fft.ifft(X)` | FFT 와 역변환(복소 입출력) |
| `np.fft.fftfreq(N, d=1/fs)` | 주파수 축(k 순서) |
| `np.fft.fftshift(X)` | k=0 을 가운데로 재배열(양측 스펙트럼을 보기 편함) |
| `np.fft.rfft(x)` / `np.fft.irfft(X, n=N)` | 실수 입력 전용, 단측만 계산 |

먼저 `fft` 와 `dft_manual` 의 결과가 수치적으로 같은지 확인한다.

In [ ]:
X_fft    = np.fft.fft(x)
freq_fft = np.fft.fftfreq(N, d=1/fs)

# 두 결과를 나란히 출력해 비교
print('k  |   freq_manual  freq_fftfreq   |   X_manual                    X_fft')
print('-' * 92)
for kk in range(N):
    fm, ff = freq_manual[kk], freq_fft[kk]
    xm, xf = X_manual[kk], X_fft[kk]
    print(f'{kk:2d} |   {fm:+8.2f}     {ff:+8.2f}   |   '
          f'{xm.real:+8.3f}{xm.imag:+7.3f}j    {xf.real:+8.3f}{xf.imag:+7.3f}j')

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
ax[0].stem(freq_manual, np.abs(X_manual), linefmt='C0-', markerfmt='C0o', basefmt=' ')
ax[0].set_xlabel('Frequency (Hz)'); ax[0].set_ylabel('|X|')
ax[0].set_title('dft_manual')
ax[1].stem(freq_fft, np.abs(X_fft), linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_title('np.fft.fft')
fig.tight_layout(); plt.show()

**관찰**: 두 결과의 값은 출력한 자릿수까지 같다. FFT 는 이 결과를 $O(N^2) \to O(N\log N)$ 으로 빠르게 계산한다.

---

## 실습 7. `fftshift` 적용 전후의 주파수 축 재배열

`np.fft.fft` 가 반환하는 순서는 `k=0..N-1`, 즉 **0 ~ fs** 를 한 바퀴 도는 순서이다. 그러나 Nyquist 주파수 이상에 해당하는 k 는 실제로는 **음의 주파수**를 뜻하므로, `fftshift` 로 **-fs/2 ~ +fs/2** 의 순서로 재배열하면 직관적으로 읽을 수 있다.

아래 두 출력의 순서 차이를 비교하라.

> 참고: 이 `|X|` 값은 아직 `/N` 으로 정규화하기 전의 스펙트럼이다. 실습 9에서 소개하는 `fft_full` 은 `/N` 정규화까지 포함한다.

In [ ]:
freq_raw     = np.fft.fftfreq(N, d=1/fs)
freq_shifted = np.fft.fftshift(freq_raw)
X_shifted    = np.fft.fftshift(X_fft)

print('BEFORE fftshift  (k=0..N-1):')
print(f'  freq = {freq_raw}')
print(f'  |X|  = {np.round(np.abs(X_fft), 3)}')
print()
print('AFTER  fftshift  (-fs/2..+fs/2):')
print(f'  freq = {freq_shifted}')
print(f'  |X|  = {np.round(np.abs(X_shifted), 3)}')

# 같은 데이터지만 배열 순서만 다름을 그래프로 확인
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
f_k = np.arange(N) * fs / N   # 정렬 전 배열 순서 그대로: k 번째 값을 k*fs/N 위치에
ax[0].stem(f_k, np.abs(X_fft), linefmt='C0-', markerfmt='C0o', basefmt=' ')
ax[0].set_xlabel('Frequency (Hz)  —  raw order k*fs/N (0..fs)'); ax[0].set_ylabel('|X|')
ax[0].set_title('BEFORE fftshift')
ax[1].stem(freq_shifted, np.abs(X_shifted), linefmt='C2-', markerfmt='C2o', basefmt=' ')
ax[1].set_xlabel('Frequency (Hz)  —  shifted (-fs/2..+fs/2)'); ax[1].set_ylabel('|X|')
ax[1].set_title('AFTER fftshift')
fig.tight_layout(); plt.show()

# 같은 데이터, 배열 순서만 다르다는 사실 코멘트
print()
print('Note: Both outputs are the same DFT; fftshift only reorders the array.')

**관찰**: 정렬하기 전의 왼쪽 패널에서는 배열 순서대로 $k f_s/N$(0 ~ 36 Hz)의 위치에 값을 그리므로, $-4$ Hz 와 $-12$ Hz 의 성분이 $f_s - f$ 인 36 Hz 와 28 Hz 에 놓인다. `fftshift` 를 적용한 오른쪽 패널에서는 DC 가 가운데로 오므로 ±4 Hz 와 ±12 Hz 의 대칭(실수 신호 $\Rightarrow |X(-f)| = |X(+f)|$)이 한눈에 보인다.

---

## 실습 8. DFT 결과의 스케일링과 물리적 진폭

`np.fft.fft` 의 출력 `|X|` 는 **샘플 수 N** 에 비례하여 커진다. 따라서 물리적 진폭으로 해석하려면 적절히 스케일링해야 한다.

같은 신호 $x(t) = 2\cos(2\pi \cdot 4 \cdot t)$ 를 두 가지 N 으로 샘플링한 결과를 다음의 세 단계로 비교한다.

1. **raw `|X|`**: 피크의 높이가 N 에 따라 커진다.
2. **`|X|/N`**: 양측 스펙트럼에서 피크의 높이가 원 신호 진폭의 **1/2**이 된다. 실수 신호의 켤레 대칭 때문에 에너지가 $\pm f$ 에 나뉜다.
3. **`2|X|/N` (단측 진폭)**: 음의 주파수의 에너지를 양의 주파수 쪽으로 합치므로 피크의 높이가 **원 신호 진폭과 일치**한다. `utils.fft` 가 이 방식을 사용한다.

In [ ]:
fs = 40
A_signal = 2.0
f0 = 4

def plot_three_scalings(N, axes_col):
    """N 샘플의 cos 신호에 대해 세 가지 스케일링을 한 컬럼 (axes_col, 길이 3) 에 그린다."""
    x = A_signal * np.cos(2*np.pi*f0*np.arange(N)/fs)

    # 양측 (fftshift 로 -fs/2 ~ +fs/2 순서 정렬)
    freq_raw = np.fft.fftfreq(N, 1/fs)
    freq     = np.fft.fftshift(freq_raw)
    X_raw    = np.fft.fft(x)
    X        = np.abs(np.fft.fftshift(X_raw))

    # 단측 2|X|/N (utils.fft)
    f_one, A_one = fft(x, fs)

    axes_col[0].stem(freq, X, linefmt='C0-', markerfmt='C0o', basefmt=' ')
    axes_col[0].set_title(f'(a) N={N}  raw |X|   peak = {X.max():.1f}')
    axes_col[0].set_ylabel('|X|')

    axes_col[1].stem(freq, X/N, linefmt='C1-', markerfmt='C1o', basefmt=' ')
    axes_col[1].set_title(f'(b) N={N}  |X|/N   peak = {(X/N).max():.2f}')
    axes_col[1].set_ylabel('|X|/N')

    axes_col[2].stem(f_one, A_one, linefmt='C2-', markerfmt='C2o', basefmt=' ')
    axes_col[2].set_title(f'(c) N={N}  one-sided 2|X|/N   peak = {A_one.max():.2f}')
    axes_col[2].set_ylabel('Amplitude'); axes_col[2].set_xlabel('Frequency (Hz)')

    return X.max(), (X/N).max(), A_one.max()


fig, axes = plt.subplots(3, 2, figsize=(13, 8))
peaks = [(N, *plot_three_scalings(N, axes[:, col]))
         for col, N in enumerate([10, 100])]
plt.tight_layout(); plt.show()

print(f'Original signal amplitude: {A_signal}')
for N, raw, norm, one in peaks:
    print(f'N={N:4d}:  raw |X| peak={raw:6.2f}   |X|/N peak={norm:.3f}   one-sided peak={one:.2f}')

**관찰**: raw `|X|` 의 피크는 $N=10$ 에서 10, $N=100$ 에서 100으로 샘플 수 $N$ 에 비례한다. `|X|/N` 로 정규화하면 두 경우 모두 피크가 1.0이며, 원 진폭 2.0의 절반이 $\pm 4$ Hz 에 하나씩 놓인다. 단측 진폭 `2|X|/N`(`utils.fft`)는 두 경우 모두 2.0으로 원 신호의 진폭과 일치한다.

---

## 실습 9. 양측 스펙트럼과 단측 스펙트럼, 공통 헬퍼 함수

**실수 신호**에서는 $X[-k] = X[k]^*$ 의 켤레 대칭이 성립하므로, 양측 스펙트럼에서 $\pm f$ 에 대칭인 성분이 나타난다.

- 양측(two-sided): $-f_s/2 \sim +f_s/2$ 의 범위에서 대칭 구조를 그대로 본다.
- 단측(one-sided): $0 \sim f_s/2$ 의 범위이다. 음의 주파수는 중복된 정보이므로 버리고 진폭을 2배로 보정한다.

공통 헬퍼 함수(`utils.py`):
- `utils.fft(x, fs) → (f, A)`: 진폭을 보정한 단측 진폭 스펙트럼
- `utils.fft_full(x, fs) → (f, X)`: `fftshift` 로 정렬하고 `1/N` 로 정규화한 양측 복소 스펙트럼

아래 셀에서 두 함수의 **내부 구현을 직접 확인**한다. 이후의 실습은 스펙트럼을 이 두 헬퍼 함수로 계산하며, 실습 11의 마스크 필터링만 `np.fft.fft` 와 `np.fft.ifft` 를 직접 사용한다.

In [ ]:
import inspect
from utils import fft, fft_full

# 헬퍼 함수 내부 확인
print('=== utils.fft (one-sided amplitude) ===')
print(inspect.getsource(fft))
print('=== utils.fft_full (two-sided complex) ===')
print(inspect.getsource(fft_full))

In [ ]:
f_2s, X_2s = fft_full(x, fs)   # 양측 복소
f_1s, A_1s = fft(x, fs)        # 단측 진폭

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].stem(f_2s, np.abs(X_2s), linefmt='C0-', markerfmt='C0o', basefmt=' ')
ax[0].set_xlabel('Frequency (Hz)'); ax[0].set_ylabel('|X|')
ax[0].set_title('Two-sided Spectrum  (fft_full)')

ax[1].stem(f_1s, A_1s, linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('Amplitude')
ax[1].set_title('One-sided Amplitude Spectrum  (fft)')
fig.tight_layout(); plt.show()

**관찰**: 오른쪽 패널의 단측 스펙트럼에서 피크의 높이는 원 코사인 신호의 진폭과 일치하며, 4 Hz 인 $f_0$ 에서 1.0, 12 Hz 인 $f_1$ 에서 0.5이다. 왼쪽 패널의 양측 스펙트럼에서는 켤레 대칭 때문에 진폭이 $\pm f$ 에 절반씩 나뉘어 나타난다.

---

## 실습 10. $f_s$ 와 관측 시간 $T$ 가 스펙트럼에 주는 영향

테스트 신호는 진폭이 2이고 주파수가 $f_0 = 0.125$ Hz 로 낮은 코사인 신호 $x(t) = 2\cos(2\pi f_0 t)$ 이다.

네 가지 $(f_s, T)$ 조합을 비교한다.

| 조합 | $f_s$ | $T$ | $N = f_s T$ | $f_s/2$ | $\Delta f = 1/T$ |
|---|---|---|---|---|---|
| Base       | 1 Hz | 8 s  | 8  | 0.5 Hz | 0.125 Hz |
| fs 증가    | 2 Hz | 8 s  | 16 | 1.0 Hz | 0.125 Hz |
| T 증가     | 1 Hz | 16 s | 16 | 0.5 Hz | 0.0625 Hz |
| 둘 다 증가 | 2 Hz | 16 s | 32 | 1.0 Hz | 0.0625 Hz |

**핵심 원리**: $f_s$ 는 **표현할 수 있는 최대 주파수**($f_s/2$)를, $T$ 는 **주파수 분해능**($\Delta f = 1/T$)을 결정하며, 두 파라미터는 서로 독립이다.

In [ ]:
def sampling_panel(fs_s, T_s, ax_top, ax_bot, title):
    N_s = int(round(fs_s * T_s))
    n_s = np.arange(N_s)
    t_s = n_s / fs_s
    x_s = 2 * np.cos(2 * np.pi * 0.125 * t_s)
    # 시간 영역: 선과 마커 (stem 대신)
    ax_top.plot(t_s, x_s, '-o', color='C0', markersize=4, linewidth=1.0)
    ax_top.set_xlim([-0.3, 16.3])
    ax_top.set_ylim([-2.3, 2.3])
    ax_top.set_xlabel('Time (s)'); ax_top.set_ylabel('x[n]')
    ax_top.set_title(title)

    # 주파수 영역: stem (이산 스펙트럼 강조)
    f_2s, X_2s = fft_full(x_s, fs_s)
    ax_bot.stem(f_2s, np.abs(X_2s), linefmt='C0-', markerfmt='C0o', basefmt=' ')
    ax_bot.set_xlim([-0.05, fs_s / 2 + 0.05])
    ax_bot.set_ylim([0, 2.3])
    ax_bot.set_xlabel('Frequency (Hz)'); ax_bot.set_ylabel('|X(f)|')
    ax_bot.set_title(f'Spectrum (0 to fs/2 = {fs_s/2:g} Hz)')

configs = [
    (1, 8,  '1 Hz , 8 sec  (Base)'),
    (2, 8,  '2 Hz , 8 sec  (Higher fs)'),
    (1, 16, '1 Hz , 16 sec  (Longer T)'),
    (2, 16, '2 Hz , 16 sec  (Both)'),
]

fig, axes = plt.subplots(4, 2, figsize=(13, 12))
for (fs_s, T_s, ttl), row in zip(configs, axes):
    sampling_panel(fs_s, T_s, row[0], row[1], ttl)
fig.tight_layout(); plt.show()

**관찰**
- 1행(Base): $f_s/2 = 0.5$ Hz, $\Delta f = 0.125$ Hz 이며, 0.125 Hz 성분이 빈 인덱스 1에 정확히 놓인다.
- 2행(Higher fs): 최대 주파수 $f_s/2$ 가 **1.0 Hz 로 증가**한다. 성분의 위치와 $\Delta f$ 는 변하지 않는다.
- 3행(Longer T): 최대 주파수는 0.5 Hz 로 그대로이고, **$\Delta f = 0.0625$ Hz 로 절반**이 된다. 빈의 개수는 2배이다.
- 4행(Both): 주파수 범위는 넓어지고 $\Delta f$ 는 작아져 두 가지가 동시에 개선된다.

**정리**: fs 는 최대 주파수에, T 는 주파수 분해능에 대응하며, 두 파라미터는 서로 독립적으로 조절할 수 있다.

---

## 실습 11. FFT 기반 필터링 (주파수 마스크)

주파수 영역에서 원하지 않는 대역을 0으로 만들고 역변환하여 필터링한다. 이 방식은 차단 특성이 급격한(sharp cutoff) 이상적 필터에 해당한다.

신호는 50 Hz 코사인에 표준편차 0.3의 가우시안 잡음을 더한 것이다. FFT 를 계산한 뒤 $\pm 50$ Hz 의 빈 두 개만 남기고 나머지를 0으로 만든다. `fs = 1000`, `N = 1000` 이므로 빈 간격이 정확히 1 Hz 이고, 마스크 조건 $|f \mp 50| < 1$ Hz 를 만족하는 빈은 $\pm 50$ Hz 에 하나씩뿐이다.

In [ ]:
fs_ex, N_ex = 1000, 1000
t_ex = np.arange(N_ex) / fs_ex
np.random.seed(0)
x_clean = np.cos(2 * np.pi * 50 * t_ex)
x_noisy = x_clean + 0.3 * np.random.randn(N_ex)

X = np.fft.fft(x_noisy)
freqs = np.fft.fftfreq(N_ex, d=1 / fs_ex)
mask = (np.abs(freqs - 50) < 1) | (np.abs(freqs + 50) < 1)
X_filt = X * mask
x_recov = np.real(np.fft.ifft(X_filt))

fig, ax = plt.subplots(3, 1, figsize=(11, 7.5))
ax[0].plot(t_ex, x_noisy, 'C0', lw=0.6)
ax[0].set_xlim([0, 0.2]); ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('x')
ax[0].set_title('(a) Noisy Signal  (cos 50 Hz + Gaussian noise)')

f_n, A_n = fft(x_noisy, fs_ex)
ax[1].plot(f_n, A_n, 'C0')
ax[1].axvline(50, color='C3', ls='--', alpha=0.6)
ax[1].set_xlim([0, 200]); ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|A|')
ax[1].set_title('(b) Noisy Spectrum  — 50 Hz peak stands out')

ax[2].plot(t_ex, x_clean, 'C1', lw=1.2, label='clean')
ax[2].plot(t_ex, x_recov, 'C2', lw=1.0, ls='--', label='recovered')
ax[2].set_xlim([0, 0.2]); ax[2].set_xlabel('Time (s)'); ax[2].set_ylabel('x')
ax[2].legend(); ax[2].set_title('(c) Clean vs Recovered  (after FFT-mask + IFFT)')
fig.tight_layout(); plt.show()

rmse = np.sqrt(np.mean((x_clean - x_recov) ** 2))
print(f'Recovery RMSE vs clean : {rmse:.4f}')

**관찰**: $\pm 50$ Hz 의 빈 두 개만 남기는 마스크로 잡음 속의 50 Hz 코사인 성분만 남긴다. `N_ex = 1000`, `fs = 1000` 이므로 빈 간격이 정확히 1 Hz 이고, 50 Hz 성분은 누설 없이 이 두 빈에만 나타난다. 출력된 복원 RMSE 는 0.0021로, 잡음이 섞인 신호의 RMSE(약 0.30, 잡음의 표준편차)의 약 140분의 1이며, (c) 패널에서는 clean 과 recovered 곡선이 거의 일치한다. 남은 오차는 두 빈에 함께 포함된 잡음에서 생기므로 난수 시드에 따라 달라진다. 시드를 바꾸면 대개 0.01 안팎이다.

---

# Part C. 합성 신호 파일에 FFT 적용

`data/data_sample_fft.csv` 는 측정 데이터가 아니라 실습용으로 만든 **합성 신호**이다. 이 파일을 불러와 시간 파형과 스펙트럼을 그린다. 데이터는 `[index, time(s), value]` 의 3개 열로 저장되어 있다.

In [ ]:
data = np.loadtxt('./data/data_sample_fft.csv', delimiter=',', skiprows=1)
t = data[:, 1]
x = data[:, 2]

dt = t[1] - t[0]
fs = 1.0 / dt
N = len(x)

print(f'N = {N},  dt = {dt:.4f} s,  fs = {fs:.1f} Hz,  T = {N*dt:.3f} s,  df = {1/(N*dt):.3f} Hz')

In [ ]:
f, A = fft(x, fs)

# scipy.signal.find_peaks 로 주요 피크 자동 검출
peak_idx, _ = find_peaks(A, height=A.max()*0.1)

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
axes[0].plot(t, x, color='C0', lw=0.8)
axes[0].set_title('(a) Time Signal  data_sample_fft.csv')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

axes[1].plot(f, A, color='C1', lw=1.2)
axes[1].plot(f[peak_idx], A[peak_idx], 'v', color='C3', markersize=8,
             label=f'find_peaks detected ({len(peak_idx)})')
axes[1].set_title('(b) One-sided Amplitude Spectrum  utils.fft(x, fs)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlim([0, fs/2])
axes[1].legend()

plt.tight_layout()
plt.show()

print('Detected peaks (Frequency, Amplitude):')
for k in peak_idx:
    print(f'  f = {f[k]:7.2f} Hz   A = {A[k]:.3f}')

- $f_s = 1000$ Hz, $N = 1000$, $T = 1$ s 이므로 $\Delta f = 1$ Hz 이다.
- 자동으로 검출된 피크는 **60 Hz** 의 단일 성분(A ≈ 2.0)이다. 이 합성 신호는 진폭이 약 2.0인 60 Hz 코사인에 표준편차가 약 0.5인 백색 잡음을 더하여 만든 것이며, 나머지 빈에 나타나는 작은 값은 그 잡음이다.

### 직접 해보기
1. 신호의 앞부분 샘플 500개만 사용하여 FFT 를 다시 계산해 보자. $\Delta f$ 는 어떻게 달라지는가?
2. `find_peaks` 의 `height` 임계값을 `A.max()*0.03` 으로 낮추면 60 Hz 외에 무엇이 검출되는가? 그것들이 실제 성분인지 잡음인지 어떻게 구분할 수 있는가?

---

# 정리

## 핵심 개념

| 개념 | 수식 / 규칙 | 요점 |
|------|------------|------|
| 샘플링 간격 | $T_s = 1/f_s$ | 초당 $f_s$ 개의 샘플 |
| 샘플링 정리 | $f_s > 2 f_{\max}$ | 위반 시 에일리어싱 발생 |
| 에일리어스 주파수 | $f_{\text{alias}} = \lvert f_0 - k f_s \rvert$ | $f_s/2$ 이하의 주파수로 나타남 |
| DFT 정의 | $X[k] = \sum_n x[n] e^{-j 2\pi k n/N}$ | $O(N^2)$ 행렬곱 |
| FFT | `np.fft.fft` | $O(N \log N)$, 결과 동일 |
| 주파수 분해능 | $\Delta f = f_s/N = 1/T$ | $N$ 또는 $T$ 증가 시 향상 |
| 단측 진폭 | $A_k = 2\lvert X[k]\rvert/N$ | DC, Nyquist 는 1배 |
| FFT 함수 규약 | `f, A = fft(x, fs)` | 이 노트북의 공통 헬퍼 함수 |

## 두 개념의 연결

샘플링은 어떤 간격으로 값을 읽을지를 정하는 시간 영역의 문제이고, DFT 는 이산 샘플을 주파수 성분으로 분해하는 주파수 영역의 문제이다. 두 개념은 다음 식으로 연결된다.

$$\boxed{\;\Delta f = \frac{1}{T} = \frac{f_s}{N}\;}$$

- 관측 시간 $T$ 를 늘리면 주파수 분해능이 향상된다.
- 샘플링 주파수 $f_s$ 를 높이면 분석할 수 있는 최대 주파수 $f_s/2$ 가 증가한다.